In [1]:
import os
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

In [2]:
import pandas as pd
import numpy as np

#df = pd.read_excel(r"C:\Users\Usuario\Downloads\Listado normas - palabra clave 1869-2023.xlsx")

df_nacional = pd.read_excel(r"C:\Users\Usuario\Downloads\Normativa nacional víctimas 1869-2023.xlsx")
df_federal = pd.read_excel(r"C:\Users\Usuario\Downloads\Catálogo federal - leyes.xlsx")

#df['caso_ok'] = np.where(df['derechos'] == "no", 0, 1)

# Elimina la columna y guarda el resultado en el mismo DataFrame
#df = df.drop(['derechos', 'texto completo', 'Resumen'], axis=1)



df_nacional_nuevo = df_nacional[['Título','caso_ok']].copy()

df_nacional_nuevo



,Título,caso_ok
0,CODIGO CIVIL,0
1,SUBSIDIO ESTATAL,1
2,SUBSIDIO ESTATAL,1
3,CODIGO PROCESAL PENAL DE LA NACION,0
4,CODIGO DE COMERCIO,0
...,...,...
199,ACUERDOS,1
200,ACUERDOS,1
201,PROGRAMA MÉDICO OBLIGATORIO DE LAS OBRAS SOCIA...,0
202,CORTE INTERAMERICANA DE DERECHOS HUMANOS,1


In [3]:
df_federal

,ID,tipo,numero,titulo,jurisdiccion,fecha,link
0,1,Ley,7379,CREACION DEL CENTRO ASISTENCIA A LA VICTIMA DE...,Córdoba,1986-02-20 00:00:00,http://www.saij.gob.ar/7379-local-cordoba-crea...
1,2,Ley,4338,Instituye día en recuerdo de las víctimas de ...,Ciudad Autónoma de Buenos Aires,2012-10-18 00:00:00,http://www.saij.gob.ar/4338-local-ciudad-auton...
2,3,Ley,5928,CREACION DEL CENTRO DE ASISTENCIA A LA VICTIMA...,Santiago del Estero,1992-10-05 00:00:00,http://www.saij.gob.ar/5928-local-santiago-est...
3,4,Ley,26738,MODIFICACION DEL CODIGO PENAL,NaN,2012-03-21 00:00:00,http://www.saij.gob.ar/26738-nacional-modifica...
4,5,Ley,1216,Creación del programa de asistencia a la vícti...,Ciudad Autónoma de Buenos Aires,2003-11-27 00:00:00,http://www.saij.gob.ar/1216-local-ciudad-auton...
...,...,...,...,...,...,...,...
1334,1335,Decreto Ley,7425,CODIGO PROCESAL CIVIL Y COMERCIAL DE BUENOS AIRES,Buenos Aires,1968-09-19 00:00:00,http://www.saij.gob.ar/7425-local-buenos-aires...
1335,1336,Ley,988,"CODIGO PROCESAL CIVIL, COMERCIAL Y MINERIA DE ...",San Juan,2014-11-19 00:00:00,http://www.saij.gob.ar/988-local-san-juan-codi...
1336,1337,Ley,2148,Código de Transito y Transporte de la CABA,Ciudad Autónoma de Buenos Aires,2006-11-16 00:00:00,http://www.saij.gob.ar/2148-local-ciudad-auton...
1337,1338,Ley,22415,CODIGO ADUANERO,NaN,1981-03-02 00:00:00,http://www.saij.gob.ar/22415-nacional-codigo-a...


In [5]:
from sklearn.model_selection import train_test_split

# 1. Definir X (todas las columnas menos la que querés predecir) e y (la columna objetivo)
# REEMPLAZA 'Nombre_De_Tu_Columna' por el nombre real en tu Excel
X = df_nacional_nuevo.drop(['Título'], axis=1) 
y = df_nacional_nuevo['caso_ok']

# 2. Dividir: el 80% para entrenar (train) y el 20% para probar (test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 3. Verificar los tamaños
print(f"Registros para entrenamiento: {len(X_train)}")
print(f"Registros para prueba: {len(X_test)}")

Registros para entrenamiento: 163
Registros para prueba: 41


In [6]:
import logging
from transformers import logging as hf_logging

# Le decimos a Hugging Face que solo muestre errores, no reportes ni warnings
hf_logging.set_verbosity_error()

from transformers import pipeline
clasificador = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [8]:
titulos = df_nacional_nuevo['Título'].astype(str).tolist()
resultados = clasificador(titulos)

In [9]:
df_nacional_nuevo['sentimiento'] = [res['label'] for res in resultados]

df_nacional_nuevo['confianza'] = [res['score'] for res in resultados]

df_nacional_nuevo

,Título,caso_ok,sentimiento,confianza
0,CODIGO CIVIL,0,NEU,0.757552
1,SUBSIDIO ESTATAL,1,NEU,0.577381
2,SUBSIDIO ESTATAL,1,NEU,0.577381
3,CODIGO PROCESAL PENAL DE LA NACION,0,NEU,0.775927
4,CODIGO DE COMERCIO,0,NEU,0.848917
...,...,...,...,...
199,ACUERDOS,1,NEU,0.516949
200,ACUERDOS,1,NEU,0.516949
201,PROGRAMA MÉDICO OBLIGATORIO DE LAS OBRAS SOCIA...,0,NEU,0.762960
202,CORTE INTERAMERICANA DE DERECHOS HUMANOS,1,NEU,0.725331


In [10]:
print(df_nacional_nuevo['sentimiento'].value_counts())
print(df_nacional_nuevo['confianza'].value_counts())

sentimiento
NEU    183
NEG     18
POS      3
Name: count, dtype: int64
confianza
0.577381    12
0.516949    10
0.628720     8
0.589180     7
0.725331     7
            ..
0.562435     1
0.672902     1
0.482219     1
0.685763     1
0.735164     1
Name: count, Length: 113, dtype: int64


In [7]:
#Análisis exploratorio de datos:

#1. Estructura y Composición
#Dimensiones: El dataset tiene al menos 205 filas (índices del 0 al 204) y originalmente varias columnas, de las cuales has filtrado las más relevantes.
#Columnas Clave:
#Titulo: Contiene nombres de leyes o normativas (ej. "CODIGO CIVIL", "SUBSIDIO ESTATAL").
#caso_ok: Una variable binaria (0 o 1) que se creó basada en la columna derechos.
#sentimiento: Etiqueta predicha por el modelo de IA (NEU para Neutro).
#confianza: El puntaje de probabilidad asignado por el modelo.
#2. Análisis de las Variables
#Variable Objetivo: caso_ok
#Se definió esta columna mediante una condición: si derechos es "no", el valor es 0; de lo contrario, es 1.
#En la muestra visible, hay una mezcla de valores, lo que indica que el dataset contiene tanto normativas que cumplen con tu criterio de "derechos" como las que no.
#Análisis de Texto y Sentimiento
#Predominancia Neutra: Según el value_counts() final, la gran mayoría de los títulos se clasifican como NEU (Neutro) con 182 registros.
#Sentimientos Minoritarios: Hay 19 registros NEG (Negativos) y solo 4 POS (Positivos). Esto es coherente, ya que los títulos de leyes suelen usar lenguaje técnico y formal, no emocional.
#Confianza del Modelo: Las puntuaciones de confianza son altas (entre 0.73 y 0.91 en la muestra), lo que sugiere que el modelo pysentimiento/robertuito-sentiment-analysis está seguro de sus clasificaciones.
#3. Distribución de Datos (Value Counts)
#En la parte inferior se observa la frecuencia de los resultados:
#Desbalance: Existe un fuerte sesgo hacia el sentimiento neutro.
#Variabilidad en Confianza: Hay una larga lista de valores únicos de confianza, lo que indica una alta granularidad en cómo el modelo interpreta cada título.

In [14]:
from huggingface_hub import notebook_login
notebook_login()


In [15]:
import torch
from transformers import pipeline, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-1B-Instruct"

# Configuración para que use MUCHA menos memoria (4 bits)
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Cargar el modelo comprimido
pipe = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"quantization_config": nf4_config},
    device_map="auto" 
)

def clasificar_ley(titulo):
    prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n¿La ley '{titulo}' trata sobre derechos de víctimas? Responde solo SI o NO.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    outputs = pipe(prompt, max_new_tokens=3, do_sample=False)
    respuesta = outputs[0]["generated_text"].split("<|assistant|>")[-1].strip().upper()
    return "SI" in respuesta

print("¡Modelo comprimido cargado con éxito!")



Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


¡Modelo comprimido cargado con éxito!


In [ ]:
from tqdm.notebook import tqdm

def clasificar_con_llama(titulo):
    if not titulo or str(titulo).lower() == 'nan':
        return "NO"
    
    # Prompt optimizado para la versión Instruct
    prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n¿La ley '{titulo}' trata sobre derechos de víctimas? Responde solo SI o NO.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    
    # Generación
    outputs = pipe(prompt, max_new_tokens=2, do_sample=False)
    
    # Extraer respuesta
    generated_text = outputs[0]["generated_text"]
    respuesta = generated_text.split("<|assistant|>")[-1].strip().upper()
    
    return "SI" if "SI" in respuesta else "NO"

# Ahora corre el proceso
resultados_nacional = []
print(f"Clasificando {len(df_nacional_nuevo)} leyes...")

for titulo in tqdm(df_nacional_nuevo['Título']):
    resultados_nacional.append(clasificar_con_llama(titulo))

df_nacional_nuevo['llama_prediccion'] = resultados_nacional
df_nacional_nuevo['llama_ok'] = df_nacional_nuevo['llama_prediccion'].apply(lambda x: 1 if x == 'SI' else 0)

# Mostrar precisión final real
from sklearn.metrics import accuracy_score
precision = accuracy_score(df_nacional_nuevo['caso_ok'], df_nacional_nuevo['llama_ok'])
print(f"Precisión real del modelo: {precision:.2%}")


Clasificando 204 leyes...


  0%|          | 0/204 [00:00<?, ?it/s]

ERROR! Session/line number was not unique in database. History logging moved to new session 10


C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cpu\ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
